In [1]:
import pandas as pd
import matplotlib.pyplot as plt
df= pd.read_csv(r"C:\Users\mahla\Downloads\DataSets\CleanedSpeakersData.csv")
print(df)

      user_id session_id sign_in                 name  demographic_age  \
0      U10477    S000001   Email  Victor Navarro-Noël               31   
1      U01536    S000002   Email                  王秀云               39   
2      U00107    S000003   Guest     Ucchal Sabharwal               68   
3      U13886    S000004   Email     Virginie Schmitt               72   
4      U05926    S000005   Email        Cynthia Drake               51   
...       ...        ...     ...                  ...              ...   
29995  U08032    S029996   Email         Radhika Gole               15   
29996  U10440    S029997   Email        Aurélie Jacob               51   
29997  U07125    S029998   Email        Rodney Miller               15   
29998  U03820    S029999   Email                  牛秀珍               73   
29999  U08963    S030000   Email                   莫超               63   

      demographic_age_group demographic_gender  \
0                     Adult             Female   
1          

In [2]:
print("="*70)
print("DATASET INFO")
print("="*70)
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"\nMissing values:")
print(df.isnull().sum())

print("\nTraffic sources:")
print(df['traffic_source'].value_counts())

print("\nConversion types:")
print(df['conversion_type'].value_counts())

DATASET INFO
Shape: (30000, 24)
Columns: ['user_id', 'session_id', 'sign_in', 'name', 'demographic_age', 'demographic_age_group', 'demographic_gender', 'email', 'location', 'country', 'device_type', 'timestamp', 'variant_group', 'time_spent', 'pages_visited', 'conversion_flag', 'conversion_type', 'traffic_source', 'product_purchased', 'revenue_$', 'payment_type', 'card_type', 'coupon_applied', 'bounce_flag']

Missing values:
user_id                  0
session_id               0
sign_in                  0
name                     0
demographic_age          0
demographic_age_group    0
demographic_gender       0
email                    0
location                 0
country                  0
device_type              0
timestamp                0
variant_group            0
time_spent               0
pages_visited            0
conversion_flag          0
conversion_type          0
traffic_source           0
product_purchased        0
revenue_$                0
payment_type             0
card

In [3]:
# Calculate funnel counts
total_sessions = len(df)
engaged_sessions = df[df['bounce_flag'] == 0].shape[0]
converted_sessions = df[df['conversion_flag'] == 1].shape[0]
purchased_sessions = df[df['conversion_type'] == 'Purchase'].shape[0]

print("\n📊 FUNNEL COUNTS:")
print(f"   Stage 1 - Sessions:        {total_sessions:>8,}")
print(f"   Stage 2 - Engaged:         {engaged_sessions:>8,}")
print(f"   Stage 3 - Converted:       {converted_sessions:>8,}")
print(f"   Stage 4 - Purchased:       {purchased_sessions:>8,}")

# Calculate drop-offs
print("\n📉 STAGE DROP-OFFS:")
drop_1_to_2 = ((total_sessions - engaged_sessions) / total_sessions) * 100
drop_2_to_3 = ((engaged_sessions - converted_sessions) / engaged_sessions) * 100
drop_3_to_4 = ((converted_sessions - purchased_sessions) / converted_sessions) * 100

print(f"   Session → Engaged:    {drop_1_to_2:.1f}% drop (bounced immediately)")
print(f"   Engaged → Converted:  {drop_2_to_3:.1f}% drop (engaged but didn't convert)")
print(f"   Converted → Purchase: {drop_3_to_4:.1f}% drop (signed up but didn't buy)")


📊 FUNNEL COUNTS:
   Stage 1 - Sessions:          30,000
   Stage 2 - Engaged:           24,327
   Stage 3 - Converted:          4,535
   Stage 4 - Purchased:          2,992

📉 STAGE DROP-OFFS:
   Session → Engaged:    18.9% drop (bounced immediately)
   Engaged → Converted:  81.4% drop (engaged but didn't convert)
   Converted → Purchase: 34.0% drop (signed up but didn't buy)


In [4]:
print("\n" + "="*70)
print("STEP 3: CHANNEL PERFORMANCE (Traffic Source Analysis)")
print("="*70)

# Group by channel
channel_summary = df.groupby('traffic_source').agg({
    'session_id': 'count',           # Total sessions
    'conversion_flag': 'sum',        # Total conversions
    'conversion_type': lambda x: (x == 'Purchase').sum(),  # Purchases only
    'revenue_$': 'sum'               # Total revenue
}).round(2)

# Calculate rates
channel_summary['conversion_rate'] = (channel_summary['conversion_flag'] / channel_summary['session_id']) * 100
channel_summary['purchase_rate'] = (channel_summary['conversion_type'] / channel_summary['session_id']) * 100

# Rename columns for clarity
channel_summary.columns = ['sessions', 'total_conversions', 'purchases', 'revenue', 'conversion_rate_%', 'purchase_rate_%']

print("\n📊 CHANNEL PERFORMANCE TABLE:")
print(channel_summary.round(1))

print("\n🎯 KEY CHANNEL INSIGHTS:")
best_channel = channel_summary['purchase_rate_%'].idxmax()
best_rate = channel_summary.loc[best_channel, 'purchase_rate_%']
print(f"   BEST channel for purchases: {best_channel} ({best_rate:.1f}%)")

most_revenue = channel_summary['revenue'].idxmax()
print(f"   HIGHEST REVENUE channel: {most_revenue} (${channel_summary.loc[most_revenue, 'revenue']:,.0f})")


STEP 3: CHANNEL PERFORMANCE (Traffic Source Analysis)

📊 CHANNEL PERFORMANCE TABLE:
                sessions  total_conversions  purchases   revenue  \
traffic_source                                                     
Organic            15084               2285       1526  412921.6   
Paid                5950                933        584  150138.4   
Referral            2974                451        309   80616.7   
Social              5992                866        573  142579.8   

                conversion_rate_%  purchase_rate_%  
traffic_source                                      
Organic                      15.1             10.1  
Paid                         15.7              9.8  
Referral                     15.2             10.4  
Social                       14.5              9.6  

🎯 KEY CHANNEL INSIGHTS:
   BEST channel for purchases: Referral (10.4%)
   HIGHEST REVENUE channel: Organic ($412,922)


In [5]:
print("\n" + "="*70)
print("STEP 4: LEAD-TO-CUSTOMER CONVERSION")
print("="*70)

# Definition: Lead = Session, Customer = Purchase
overall_rate = (purchased_sessions / total_sessions) * 100
print(f"\n📈 OVERALL LEAD-TO-CUSTOMER RATE: {overall_rate:.2f}%")
print(f"   (Out of {total_sessions:,} sessions, {purchased_sessions:,} resulted in purchases)")

print("\n📊 LEAD-TO-CUSTOMER BY CHANNEL:")
for channel in df['traffic_source'].unique():
    channel_df = df[df['traffic_source'] == channel]
    channel_sessions = len(channel_df)
    channel_purchases = channel_df[channel_df['conversion_type'] == 'Purchase'].shape[0]
    channel_rate = (channel_purchases / channel_sessions) * 100
    
    # Also calculate signups
    channel_signups = channel_df[channel_df['conversion_type'] == 'Signup'].shape[0]
    signup_rate = (channel_signups / channel_sessions) * 100
    
    print(f"\n   {channel.upper()}:")
    print(f"      Sessions: {channel_sessions:,}")
    print(f"      Customers (Purchases): {channel_purchases} ({channel_rate:.2f}%)")
    print(f"      Signups: {channel_signups} ({signup_rate:.2f}%)")

# Best channel for lead→customer
best_purchase_channel = df[df['conversion_type'] == 'Purchase'].groupby('traffic_source').size().idxmax()
best_purchase_rate = (df[(df['traffic_source'] == best_purchase_channel) & (df['conversion_type'] == 'Purchase')].shape[0] / 
                      len(df[df['traffic_source'] == best_purchase_channel])) * 100

print(f"\n🏆 BEST CHANNEL FOR LEAD→CUSTOMER: {best_purchase_channel}")
print(f"   Converts {best_purchase_rate:.1f}% of sessions into purchases")


STEP 4: LEAD-TO-CUSTOMER CONVERSION

📈 OVERALL LEAD-TO-CUSTOMER RATE: 9.97%
   (Out of 30,000 sessions, 2,992 resulted in purchases)

📊 LEAD-TO-CUSTOMER BY CHANNEL:

   ORGANIC:
      Sessions: 15,084
      Customers (Purchases): 1526 (10.12%)
      Signups: 759 (5.03%)

   SOCIAL:
      Sessions: 5,992
      Customers (Purchases): 573 (9.56%)
      Signups: 293 (4.89%)

   PAID:
      Sessions: 5,950
      Customers (Purchases): 584 (9.82%)
      Signups: 349 (5.87%)

   REFERRAL:
      Sessions: 2,974
      Customers (Purchases): 309 (10.39%)
      Signups: 142 (4.77%)

🏆 BEST CHANNEL FOR LEAD→CUSTOMER: Organic
   Converts 10.1% of sessions into purchases


In [6]:
print("\n" + "="*70)
print("STEP 5: DEVICE & ENGAGEMENT ANALYSIS")
print("="*70)

# Device performance
device_summary = df.groupby('device_type').agg({
    'session_id': 'count',
    'conversion_flag': 'sum',
    'conversion_type': lambda x: (x == 'Purchase').sum(),
    'pages_visited': 'mean',
    'bounce_flag': 'mean'
}).round(2)

device_summary['conversion_rate'] = (device_summary['conversion_flag'] / device_summary['session_id']) * 100
device_summary['purchase_rate'] = (device_summary['conversion_type'] / device_summary['session_id']) * 100
device_summary['bounce_rate'] = device_summary['bounce_flag'] * 100

print("\n📊 DEVICE PERFORMANCE:")
print(device_summary[['session_id', 'purchase_rate', 'pages_visited', 'bounce_rate']])

# Pages visited analysis
print("\n📄 PAGES VISITED VS CONVERSION:")
for pages in [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]:
    pages_df = df[df['pages_visited'] == pages]
    if len(pages_df) > 0:
        conversion_pct = (pages_df['conversion_flag'].sum() / len(pages_df)) * 100
        print(f"   {pages:2d} pages: {len(pages_df):6,} sessions → {conversion_pct:.1f}% conversion")


STEP 5: DEVICE & ENGAGEMENT ANALYSIS

📊 DEVICE PERFORMANCE:
             session_id  purchase_rate  pages_visited  bounce_rate
device_type                                                       
Desktop            8953       9.985480           5.50         19.0
Mobile            18008       9.923367           5.51         19.0
Tablet             3039      10.233629           5.48         19.0

📄 PAGES VISITED VS CONVERSION:
    1 pages:  3,010 sessions → 15.9% conversion
    2 pages:  3,007 sessions → 15.5% conversion
    3 pages:  2,965 sessions → 15.1% conversion
    4 pages:  2,969 sessions → 14.2% conversion
    5 pages:  3,059 sessions → 15.3% conversion
    6 pages:  3,044 sessions → 15.5% conversion
    7 pages:  2,942 sessions → 14.7% conversion
    8 pages:  2,950 sessions → 14.5% conversion
    9 pages:  3,044 sessions → 14.8% conversion
   10 pages:  3,010 sessions → 15.7% conversion


In [7]:
print("\n" + "="*70)
print("STEP 6: DEMOGRAPHIC ANALYSIS")
print("="*70)

# Age group performance
age_summary = df.groupby('demographic_age_group').agg({
    'session_id': 'count',
    'conversion_flag': 'sum',
    'conversion_type': lambda x: (x == 'Purchase').sum(),
    'revenue_$': 'sum'
}).round(2)

age_summary['purchase_rate'] = (age_summary['conversion_type'] / age_summary['session_id']) * 100

print("\n👥 CONVERSION BY AGE GROUP:")
print(age_summary[['session_id', 'purchase_rate', 'revenue_$']])

# Gender analysis
gender_summary = df.groupby('demographic_gender').agg({
    'session_id': 'count',
    'conversion_flag': 'sum',
    'conversion_type': lambda x: (x == 'Purchase').sum(),
    'revenue_$': 'sum'
}).round(2)

gender_summary['purchase_rate'] = (gender_summary['conversion_type'] / gender_summary['session_id']) * 100

print("\n👤 CONVERSION BY GENDER:")
print(gender_summary[['session_id', 'purchase_rate', 'revenue_$']])

# Best segment
best_age = age_summary['purchase_rate'].idxmax()
best_gender = gender_summary['purchase_rate'].idxmax()
print(f"\n🎯 BEST PERFORMING SEGMENT: {best_age} {best_gender}")


STEP 6: DEMOGRAPHIC ANALYSIS

👥 CONVERSION BY AGE GROUP:
                       session_id  purchase_rate  revenue_$
demographic_age_group                                      
Adult                       20139       9.926014  507457.60
Old                          7156      10.131358  190916.86
Teenage                      2705       9.907579   87882.13

👤 CONVERSION BY GENDER:
                    session_id  purchase_rate  revenue_$
demographic_gender                                      
Female                   13487       9.965152  343260.67
Male                     13606       9.907394  365393.08
No Answer                 2907      10.319917   77602.84

🎯 BEST PERFORMING SEGMENT: Old No Answer


In [8]:
print("\n" + "="*70)
print("STEP 7: COUPON & PAYMENT ANALYSIS")
print("="*70)

# Coupon effectiveness
coupon_summary = df.groupby('coupon_applied').agg({
    'session_id': 'count',
    'conversion_flag': 'sum',
    'conversion_type': lambda x: (x == 'Purchase').sum(),
    'revenue_$': 'mean'
}).round(2)

coupon_summary['purchase_rate'] = (coupon_summary['conversion_type'] / coupon_summary['session_id']) * 100

print("\n🏷️ COUPON IMPACT:")
print(coupon_summary[['session_id', 'purchase_rate', 'revenue_$']])

# Payment type analysis
payment_summary = df[df['payment_type'] != 'NPT'].groupby('payment_type').agg({
    'session_id': 'count',
    'revenue_$': 'sum'
}).round(2)

print("\n💳 PAYMENT METHOD PREFERENCE:")
print(payment_summary)


STEP 7: COUPON & PAYMENT ANALYSIS

🏷️ COUPON IMPACT:
                session_id  purchase_rate  revenue_$
coupon_applied                                      
ND                   27008            0.0        0.0
No                    2126          100.0      261.6
Yes                    866          100.0      265.7

💳 PAYMENT METHOD PREFERENCE:
              session_id  revenue_$
payment_type                       
COD                  897  219927.56
Card                2095  566329.03


In [10]:
print("\n" + "="*70)
print("STEP 8: RECOMMENDATIONS TO IMPROVE LEAD→CUSTOMER")
print("="*70)

# Find biggest opportunities
print("\n📊 KEY FINDINGS:")
print(f"   1. Best channel: {best_channel} ({best_rate:.1f}% purchase rate)")
print(f"   2. Best device: {best_device if 'best_device' in dir() else 'Desktop'}")

# Recommendations
print("\n💡 RECOMMENDATIONS:")
print("-"*50)

# Channel recommendation
worst_channel = channel_summary['purchase_rate_%'].idxmin()
print(f"1. CHANNEL OPTIMIZATION:")
print(f"   → Increase budget on {best_channel} (best ROI)")
print(f"   → Investigate why {worst_channel} underperforms")

# Device recommendation
mobile_rate = device_summary.loc['Mobile', 'purchase_rate'] if 'Mobile' in device_summary.index else 0
desktop_rate = device_summary.loc['Desktop', 'purchase_rate'] if 'Desktop' in device_summary.index else 0
if mobile_rate < desktop_rate:
    print(f"\n2. MOBILE OPTIMIZATION:")
    print(f"   → Mobile converts at {mobile_rate:.1f}% vs Desktop {desktop_rate:.1f}%")
    print(f"   → Optimize mobile checkout experience")

# Coupon recommendation
coupon_no = coupon_summary.loc['No', 'purchase_rate'] if 'No' in coupon_summary.index else 0
coupon_yes = coupon_summary.loc['Yes', 'purchase_rate'] if 'Yes' in coupon_summary.index else 0
if coupon_yes > coupon_no:
    print(f"\n3. COUPON STRATEGY:")
    print(f"   → Coupons increase conversion ({coupon_yes:.1f}% vs {coupon_no:.1f}%)")
    print(f"   → Test limited-time coupon campaigns")

print(f"\n4. LEAD-TO-CUSTOMER IMPROVEMENT:")
print(f"   → Current rate: {overall_rate:.2f}%")
print(f"   → Target: {overall_rate * 1.3:.2f}% (+30% improvement)")
print(f"   → Strategy: Implement cart abandonment emails + mobile optimization")


STEP 8: RECOMMENDATIONS TO IMPROVE LEAD→CUSTOMER

📊 KEY FINDINGS:
   1. Best channel: Referral (10.4% purchase rate)
   2. Best device: Desktop

💡 RECOMMENDATIONS:
--------------------------------------------------
1. CHANNEL OPTIMIZATION:
   → Increase budget on Referral (best ROI)
   → Investigate why Social underperforms

2. MOBILE OPTIMIZATION:
   → Mobile converts at 9.9% vs Desktop 10.0%
   → Optimize mobile checkout experience

4. LEAD-TO-CUSTOMER IMPROVEMENT:
   → Current rate: 9.97%
   → Target: 12.97% (+30% improvement)
   → Strategy: Implement cart abandonment emails + mobile optimization


In [11]:
print("\n" + "="*70)
print("STEP: GEOGRAPHIC PERFORMANCE ANALYSIS")
print("="*70)

print("\n🌍 TOP PERFORMING COUNTRIES:")
print("-"*50)

# Country performance
country_summary = df.groupby('country').agg({
    'session_id': 'count',
    'conversion_flag': 'sum',
    'conversion_type': lambda x: (x == 'Purchase').sum(),
    'revenue_$': 'sum'
}).round(2)

country_summary['purchase_rate'] = (country_summary['conversion_type'] / country_summary['session_id']) * 100
country_summary['conversion_rate'] = (country_summary['conversion_flag'] / country_summary['session_id']) * 100

# Sort by purchase rate (best performing countries)
country_best = country_summary.nlargest(10, 'purchase_rate')

print("\n📊 TOP 10 COUNTRIES BY PURCHASE RATE:")
print(country_best[['session_id', 'purchase_rate', 'revenue_$']])

# Countries with most revenue
print("\n💰 TOP 10 COUNTRIES BY REVENUE:")
country_revenue = country_summary.nlargest(10, 'revenue_$')
print(country_revenue[['session_id', 'purchase_rate', 'revenue_$']])


STEP: GEOGRAPHIC PERFORMANCE ANALYSIS

🌍 TOP PERFORMING COUNTRIES:
--------------------------------------------------

📊 TOP 10 COUNTRIES BY PURCHASE RATE:
           session_id  purchase_rate  revenue_$
country                                        
China            2272      11.355634   71220.65
Spain            1174      10.988075   36299.58
France           2290      10.960699   72268.47
UAE              1147      10.897995   31992.53
Germany          2274      10.290237   56441.91
UK               2324       9.896730   53471.83
Singapore        1116       9.856631   28583.08
Italy            1148       9.756098   27715.45
USA              3444       9.756098   99698.45
India            3529       9.719467   83348.92

💰 TOP 10 COUNTRIES BY REVENUE:
           session_id  purchase_rate  revenue_$
country                                        
USA              3444       9.756098   99698.45
India            3529       9.719467   83348.92
France           2290      10.960699   7226

In [12]:
print("\n" + "="*70)
print("📊 CITY-LEVEL PERFORMANCE")
print("="*70)

# Top 20 cities by performance
city_summary = df.groupby('location').agg({
    'session_id': 'count',
    'conversion_flag': 'sum',
    'conversion_type': lambda x: (x == 'Purchase').sum(),
    'revenue_$': 'sum'
}).round(2)

city_summary['purchase_rate'] = (city_summary['conversion_type'] / city_summary['session_id']) * 100

# Filter cities with at least 20 sessions for reliability
city_reliable = city_summary[city_summary['session_id'] >= 20]

print("\n🏆 TOP 10 CITIES WITH BEST CONVERSION RATES:")
city_best = city_reliable.nlargest(10, 'purchase_rate')
for city, row in city_best.iterrows():
    print(f"   {city:20s}: {row['session_id']:4.0f} sessions → {row['purchase_rate']:.1f}% purchase rate (${row['revenue_$']:,.0f} revenue)")

print("\n💰 TOP 10 CITIES BY REVENUE:")
city_revenue = city_reliable.nlargest(10, 'revenue_$')
for city, row in city_revenue.iterrows():
    print(f"   {city:20s}: ${row['revenue_$']:>10,.0f} revenue ({row['session_id']:4.0f} sessions, {row['purchase_rate']:.1f}% conversion)")


📊 CITY-LEVEL PERFORMANCE

🏆 TOP 10 CITIES WITH BEST CONVERSION RATES:
   Shanghai            : 1158 sessions → 12.2% purchase rate ($38,153 revenue)
   Lyon                : 1131 sessions → 11.7% purchase rate ($35,440 revenue)
   Madrid              : 1174 sessions → 11.0% purchase rate ($36,300 revenue)
   Dubai               : 1147 sessions → 10.9% purchase rate ($31,993 revenue)
   Berlin              : 1138 sessions → 10.9% purchase rate ($30,559 revenue)
   New York            : 1114 sessions → 10.9% purchase rate ($32,883 revenue)
   Beijing             : 1114 sessions → 10.5% purchase rate ($33,068 revenue)
   Paris               : 1159 sessions → 10.3% purchase rate ($36,828 revenue)
   Manchester          : 1200 sessions → 10.2% purchase rate ($30,587 revenue)
   Bangalore           : 1153 sessions → 10.1% purchase rate ($24,835 revenue)

💰 TOP 10 CITIES BY REVENUE:
   Shanghai            : $    38,153 revenue (1158 sessions, 12.2% conversion)
   Paris               : $    3

In [13]:
print("\n" + "="*70)
print("📊 CITIES-LEVEL PERFOMANCE")
print("="*70)

# Most active cities (most sessions)
print("\n📍 MOST ACTIVE CITIES (Highest traffic):")
city_traffic = city_summary.nlargest(10, 'session_id')
for city, row in city_traffic.iterrows():
    print(f"   {city:18s}: {row['session_id']:4.0f} sessions → {row['purchase_rate']:.1f}% purchase rate")

# Identify underperforming high-traffic cities
print("\n⚠️ HIGH TRAFFIC, LOW CONVERSION CITIES (Opportunities):")
city_high_traffic_low_conv = city_reliable[
    (city_reliable['session_id'] > city_reliable['session_id'].median()) & 
    (city_reliable['purchase_rate'] < city_reliable['purchase_rate'].median())
].nlargest(5, 'session_id')

for city, row in city_high_traffic_low_conv.iterrows():
    print(f"   {city:18s}: {row['session_id']:4.0f} sessions, {row['purchase_rate']:.1f}% conversion (needs improvement!)")


📊 REGIONAL PERFOMANCE

📍 MOST ACTIVE CITIES (Highest traffic):
   Manchester        : 1200 sessions → 10.2% purchase rate
   Chicago           : 1199 sessions → 9.1% purchase rate
   Toronto           : 1194 sessions → 9.8% purchase rate
   Mumbai            : 1191 sessions → 9.8% purchase rate
   Delhi             : 1185 sessions → 9.3% purchase rate
   Vancouver         : 1180 sessions → 9.6% purchase rate
   Hong Kong         : 1176 sessions → 8.4% purchase rate
   Madrid            : 1174 sessions → 11.0% purchase rate
   Osaka             : 1170 sessions → 9.8% purchase rate
   Sydney            : 1170 sessions → 9.7% purchase rate

⚠️ HIGH TRAFFIC, LOW CONVERSION CITIES (Opportunities):
   Chicago           : 1199 sessions, 9.1% conversion (needs improvement!)
   Toronto           : 1194 sessions, 9.8% conversion (needs improvement!)
   Delhi             : 1185 sessions, 9.3% conversion (needs improvement!)
   Vancouver         : 1180 sessions, 9.6% conversion (needs improvement

In [14]:
print("\n" + "="*70)
print("📊 CONTINENT/REGION ANALYSIS")
print("="*70)

# Define continent mapping
continent_map = {
    'Australia': 'Oceania', 'New Zealand': 'Oceania',
    'United States': 'North America', 'Canada': 'North America', 'Mexico': 'North America',
    'United Kingdom': 'Europe', 'France': 'Europe', 'Germany': 'Europe', 'Spain': 'Europe', 
    'Italy': 'Europe', 'Netherlands': 'Europe', 'Sweden': 'Europe', 'Norway': 'Europe',
    'China': 'Asia', 'India': 'Asia', 'Japan': 'Asia', 'Singapore': 'Asia', 'Hong Kong': 'Asia',
    'Brazil': 'South America', 'Argentina': 'South America',
    'South Africa': 'Africa', 'Egypt': 'Africa'
}

df['continent'] = df['country'].map(continent_map).fillna('Other')

# Continent performance
continent_summary = df.groupby('continent').agg({
    'session_id': 'count',
    'conversion_type': lambda x: (x == 'Purchase').sum(),
    'revenue_$': 'sum'
}).round(2)

continent_summary['purchase_rate'] = (continent_summary['conversion_type'] / continent_summary['session_id']) * 100

print("\n🌎 PERFORMANCE BY CONTINENT:")
continent_summary = continent_summary.sort_values('purchase_rate', ascending=False)
for continent, row in continent_summary.iterrows():
    bar = '█' * int(row['purchase_rate'])
    print(f"   {continent:15s}: {row['purchase_rate']:5.1f}% {bar} ({row['session_id']:4.0f} sessions, ${row['revenue_$']:>8,.0f} revenue)")


📊 CONTINENT/REGION ANALYSIS

🌎 PERFORMANCE BY CONTINENT:
   Europe         :  10.5% ██████████ (6886 sessions, $ 192,725 revenue)
   Asia           :   9.9% █████████ (10416 sessions, $ 258,185 revenue)
   Other          :   9.9% █████████ (8013 sessions, $ 212,406 revenue)
   North America  :   9.7% █████████ (2374 sessions, $  66,583 revenue)
   Oceania        :   9.3% █████████ (2311 sessions, $  56,357 revenue)
